# Distributed Training at Scale

Companion notebook for the [Distributed Training lesson](https://ml-viz-ruby.vercel.app/courses/gpu-programming/05-distributed-training-at-scale).

**The idea in one sentence.** A model that doesn't fit — or won't train fast enough —
on one GPU is split across many, and the four strategies trade **communication** for
**memory**: **data parallelism** (replicate, all-reduce gradients), **ZeRO/FSDP**
(shard the optimizer state), **pipeline** and **tensor** parallelism (split the model
itself).

What this notebook quantifies from scratch:

- **Data parallelism** — ring all-reduce traffic per GPU approaches a *constant* as
  you add GPUs (it scales!).
- **The memory wall** — mixed-precision Adam needs $16\Psi$ bytes per parameter,
  which blows past one GPU fast.
- **ZeRO stages** — sharding the state shrinks per-GPU memory toward $16\Psi/G$.
- **Pipeline bubble** — idle time that more micro-batches amortise.

We **validate the all-reduce scaling, the ZeRO-3 memory factor, and the bubble
formula**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})

## 1 — Data parallelism: the all-reduce cost barely grows with G

A ring all-reduce moves roughly `2*Psi*(G-1)/G` bytes of gradient data per GPU, where `Psi` is the
parameter count and `G` the number of GPUs. As `G` grows, `(G-1)/G -> 1`, so the per-GPU
communication cost approaches a *constant* — it does not keep growing with G. This is why data
parallelism scales well, provided the all-reduce overlaps with backward compute.

In [ ]:
def ring_allreduce_bytes_per_gpu(psi_params, G, bytes_per_param=2):
    """Approx bytes moved per GPU in a ring all-reduce of the gradient tensor."""
    total_bytes = psi_params * bytes_per_param
    return total_bytes * 2 * (G - 1) / G

psi = 7e9  # 7B parameters
Gs = np.array([2, 4, 8, 16, 32, 64, 128, 256])
per_gpu_gb = [ring_allreduce_bytes_per_gpu(psi, G) / 1e9 for G in Gs]

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.semilogx(Gs, per_gpu_gb, 'o-', color='#2dd4bf', base=2)
ax.set_xlabel('number of GPUs (G)'); ax.set_ylabel('all-reduce traffic per GPU (GB)')
ax.set_title('Per-GPU all-reduce traffic approaches a constant as G grows')
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout(); plt.show()

print(f"G=2:   {per_gpu_gb[0]:.2f} GB/GPU")
print(f"G=32:  {per_gpu_gb[4]:.2f} GB/GPU")
print(f"G=256: {per_gpu_gb[-1]:.2f} GB/GPU  (barely more than G=32 -- the curve has flattened)")

### Validate: ring all-reduce traffic per GPU is (almost) constant in $G$

A ring all-reduce moves $2(G-1)/G$ times the gradient size per GPU, which
$\to 2\times$ the gradient as $G\to\infty$ — *independent of $G$*. That constant
per-GPU cost is exactly why data parallelism scales to thousands of GPUs. We confirm
the traffic saturates rather than growing.

In [ ]:
bytes_grad = psi * 2
traffic = [ring_allreduce_bytes_per_gpu(psi, G) for G in Gs]
for G, t in zip(Gs, traffic):
    print(f'G={G:3d}: per-GPU all-reduce = {t/1e9:.2f} GB  ({t/bytes_grad:.3f} x gradient size)')
assert traffic[-1] > traffic[0], 'traffic rises slightly then saturates'
assert all(t < 2 * bytes_grad for t in traffic), 'per-GPU traffic is bounded by 2x the gradient'
assert abs(traffic[-1] - 2 * bytes_grad) < 0.05 * (2 * bytes_grad), 'it approaches the 2x constant'
print('\n✅ per-GPU communication approaches a constant 2x the gradient — data parallelism scales')

## 2 — The memory wall: 16Ψ bytes per parameter

Mixed-precision Adam training keeps, per parameter: fp16 weights (2 bytes), fp16 gradients
(2 bytes), and fp32 optimizer state — master weights + momentum + variance (4+4+4 = 12 bytes).
That's **16 bytes per parameter**, before a single activation is stored.

In [ ]:
def model_state_bytes(num_params):
    fp16_params = 2 * num_params
    fp16_grads = 2 * num_params
    fp32_optimizer_state = 12 * num_params  # master weights + momentum + variance
    return fp16_params, fp16_grads, fp32_optimizer_state

for n_params, name in [(7e9, '7B'), (70e9, '70B')]:
    p, g, o = model_state_bytes(n_params)
    total_gb = (p + g + o) / 1e9
    print(f"{name} model: params={p/1e9:.1f}GB  grads={g/1e9:.1f}GB  optimizer={o/1e9:.1f}GB  "
          f"-> total={total_gb:.1f}GB per GPU under naive data parallelism")

## 3 — ZeRO / FSDP: sharding shrinks per-GPU memory with G

Each ZeRO stage shards one more component of the 16Ψ bytes across the `G` data-parallel GPUs:

| Stage | Per-GPU bytes |
|---|---|
| Baseline DP | 16Ψ |
| ZeRO-1 (shard optimizer state) | 4Ψ + 12Ψ/G |
| ZeRO-2 (+ shard gradients) | 2Ψ + 14Ψ/G |
| ZeRO-3 (+ shard parameters) | 16Ψ/G |

In [ ]:
def zero_stage_bytes_per_gpu(num_params, G, stage):
    """Per-GPU model-state bytes for ZeRO stage 0 (baseline DP), 1, 2, or 3."""
    psi = num_params
    if stage == 0:
        return 16 * psi
    if stage == 1:
        return 4 * psi + 12 * psi / G
    if stage == 2:
        return 2 * psi + 14 * psi / G
    if stage == 3:
        return 16 * psi / G
    raise ValueError("stage must be 0, 1, 2, or 3")

psi = 7e9
Gs = np.array([1, 2, 4, 8, 16, 32, 64])
fig, ax = plt.subplots(figsize=(8, 4.5))
colors = {0: '#f43f5e', 1: '#eab308', 2: '#818cf8', 3: '#2dd4bf'}
labels = {0: 'Baseline DP', 1: 'ZeRO-1', 2: 'ZeRO-2', 3: 'ZeRO-3'}
for stage in [0, 1, 2, 3]:
    gb = [zero_stage_bytes_per_gpu(psi, G, stage) / 1e9 for G in Gs]
    ax.plot(Gs, gb, 'o-', color=colors[stage], label=labels[stage])
ax.axhline(80, ls='--', color='#94a3b8', label='80GB GPU')
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xlabel('GPUs in data-parallel group (G)'); ax.set_ylabel('per-GPU model state (GB)')
ax.set_title('7B-parameter model: per-GPU memory by ZeRO stage')
ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout(); plt.show()

for stage in [0, 1, 2, 3]:
    gb = zero_stage_bytes_per_gpu(psi, 8, stage) / 1e9
    print(f"{labels[stage]:12s} @ G=8: {gb:6.1f} GB/GPU")

### Validate: ZeRO-3 cuts per-GPU memory to $16\Psi/G$

Baseline data parallelism replicates all $16\Psi$ bytes of model state on every GPU.
ZeRO progressively shards it: stage 3 shards *everything*, so per-GPU memory is
$16\Psi/G$ — a factor-of-$G$ reduction. We confirm each stage shrinks with $G$ and
ZeRO-3 hits the $1/G$ ideal.

In [ ]:
base = zero_stage_bytes_per_gpu(psi, 1, 0)     # 16*psi, independent of G
for G in [1, 2, 8, 64]:
    z3 = zero_stage_bytes_per_gpu(psi, G, 3)
    print(f'G={G:3d}: baseline {base/1e9:.0f}GB  ZeRO-3 {z3/1e9:6.2f}GB  (factor {base/z3:.0f} smaller)')
    assert np.isclose(z3, 16 * psi / G), 'ZeRO-3 per-GPU memory is 16*psi/G'
# ZeRO stages are monotone: 0 (most memory) >= 1 >= 2 >= 3 (least) for G>1
for G in [8, 64]:
    mems = [zero_stage_bytes_per_gpu(psi, G, s) for s in [0, 1, 2, 3]]
    assert mems[0] >= mems[1] >= mems[2] >= mems[3], 'higher ZeRO stage shards more'
print('\n✅ ZeRO-3 shards all state to 16*psi/G — sharding trades communication for memory')

## 4 — Pipeline parallelism: the idle bubble

Splitting the model across `P` pipeline stages and running `M` micro-batches through them leaves
the pipeline idle for a fraction of the time given by `bubble_fraction = (P-1) / (P-1+M)`. More
micro-batches shrink the bubble for a fixed number of stages.

In [ ]:
def pipeline_bubble_fraction(P, M):
    return (P - 1) / (P - 1 + M)

P = 5
Ms = np.array([1, 2, 4, 8, 15, 32, 64, 128])
bubble = [pipeline_bubble_fraction(P, M) for M in Ms]

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.semilogx(Ms, bubble, 'o-', color='#f97316', base=2)
ax.set_xlabel('micro-batches (M)'); ax.set_ylabel('bubble fraction (idle time)')
ax.set_title(f'Pipeline bubble shrinks with more micro-batches (P = {P} stages)')
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout(); plt.show()

print(f"P={P}, M=1:   bubble = {pipeline_bubble_fraction(P, 1):.2f}  (80% idle!)")
print(f"P={P}, M=15:  bubble = {pipeline_bubble_fraction(P, 15):.2f}")
print(f"P={P}, M=128: bubble = {pipeline_bubble_fraction(P, 128):.3f}")

### Validate: the pipeline bubble vanishes with more micro-batches

Pipeline parallelism idles $(P-1)$ stages while the pipeline fills and drains; the
bubble fraction $(P-1)/(P-1+M)$ shrinks toward 0 as the number of micro-batches $M$
grows. We confirm it decreases monotonically and approaches zero.

In [ ]:
P = 5
fracs = [pipeline_bubble_fraction(P, M) for M in Ms]
for M, f in zip(Ms, fracs):
    print(f'P={P}, M={M:3d}: bubble = {f:.3f} ({f:.0%} idle)')
assert all(fracs[i] > fracs[i+1] for i in range(len(fracs)-1)), 'more micro-batches shrink the bubble'
assert fracs[-1] < 0.05, 'with enough micro-batches the pipeline is nearly always busy'
print('\n✅ the pipeline bubble amortises away with micro-batching — keep P small, M large')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **communication-bound** | all-reduce/all-gather must overlap with compute or the GPUs starve |
| **ZeRO-3 comms overhead** | sharding everything adds all-gather traffic each step — memory-for-bandwidth |
| **pipeline bubble** | too few micro-batches idles most of the pipeline (demo); need $M\gg P$ |
| **activation memory** | many in-flight micro-batches raise activation memory; use recomputation |
| **load imbalance** | uneven pipeline stages stall the whole line at the slowest one |

Demo: deeper pipelines need many more micro-batches to keep the bubble small.

In [ ]:
# The catch for pipeline parallelism: you need M >> P to hide the bubble, but larger M
# means more activations kept in flight (more memory). And every strategy has a
# comms cost the compute must overlap. We show the bubble's sensitivity to depth P.
for P in [2, 4, 8, 16]:
    M_for_10pct = None
    for M in range(1, 100000):
        if pipeline_bubble_fraction(P, M) <= 0.10:
            M_for_10pct = M; break
    print(f'P={P:2d} stages: need M >= {M_for_10pct} micro-batches to get the bubble under 10%')
print('\nDeeper pipelines need many more micro-batches to stay efficient -> real systems')
print('combine data + tensor + pipeline parallelism (3D parallelism) to balance the tradeoffs.')

## ✏️ Your turn

**Exercise.** Implement `zero_stage_memory_fraction(stage, G)` — the per-GPU memory at a given
ZeRO stage as a *fraction of the 16Ψ-byte baseline* (so it's independent of parameter count) —
and `pipeline_bubble_fraction(P, M)` (same formula as above, reimplemented from scratch).

In [ ]:
def zero_stage_memory_fraction(stage, G):
    # TODO(you): per-GPU bytes at this stage, divided by the 16*psi baseline.
    # Reuse the stage formulas from part 3, then divide by 16 (psi cancels out).
    return ...

def pipeline_bubble_fraction(P, M):
    # TODO(you): (P - 1) / (P - 1 + M)
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert zero_stage_memory_fraction(0, 8) == 1.0             # baseline: no sharding
assert abs(zero_stage_memory_fraction(3, 8) - 1 / 8) < 1e-9  # ZeRO-3: fully sharded, 16/G / 16 = 1/G
assert zero_stage_memory_fraction(3, 1) == 1.0              # G=1: sharding across 1 GPU changes nothing
assert abs(pipeline_bubble_fraction(5, 15) - 4 / 19) < 1e-9
assert pipeline_bubble_fraction(2, 1) == 0.5
print("✓ ZeRO fraction + pipeline bubble checks pass")

<details>
<summary>Solution</summary>

```python
def zero_stage_memory_fraction(stage, G):
    psi = 1.0  # work in units of Psi so the fraction is parameter-count independent
    if stage == 0:
        bytes_per_gpu = 16 * psi
    elif stage == 1:
        bytes_per_gpu = 4 * psi + 12 * psi / G
    elif stage == 2:
        bytes_per_gpu = 2 * psi + 14 * psi / G
    elif stage == 3:
        bytes_per_gpu = 16 * psi / G
    else:
        raise ValueError("stage must be 0, 1, 2, or 3")
    return bytes_per_gpu / (16 * psi)

def pipeline_bubble_fraction(P, M):
    return (P - 1) / (P - 1 + M)
```

ZeRO-3's fraction is exactly `1/G` — per-GPU memory scales down linearly with the number of GPUs,
which is the whole point of full sharding. The pipeline bubble fraction shows why real pipeline-parallel
training always uses many small micro-batches: with only `M=1`, three quarters of a 5-stage pipeline's
time is wasted waiting.

</details>

## Key takeaways

- **Data parallelism scales** because ring all-reduce per-GPU traffic approaches a
  *constant* ($2\times$ the gradient), independent of $G$ (verified).
- **The memory wall is real:** mixed-precision Adam needs $16\Psi$ bytes/param, far
  exceeding one GPU for large models.
- **ZeRO/FSDP shards the state:** stage 3 gives $16\Psi/G$ per GPU (verified) —
  trading extra communication for a factor-$G$ memory cut.
- **Pipeline parallelism has a bubble** $(P-1)/(P-1+M)$ that micro-batching amortises
  (verified); deeper pipelines need much larger $M$ (demo).
- **Real systems combine all three (3D parallelism)** to balance memory and comms.